In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd


import os
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt

import seaborn as sns
import shap
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

warnings.filterwarnings("ignore")



from pathlib import Path
prep_path = Path(r"C:\Users\pradi\Pradip Workspace\The IOT acadamy assignment\AI ML Internship Programe\Telco Customer Churn\ipynb file\03_preprocessing.ipynb")
print(f"Running preprocessing notebook: {prep_path}")
%run "{prep_path}"



# Create pipeline dictionary with 5 models and class_weight='balanced'
models = {
    'Logistic Regression': Pipeline([
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    'KNN': Pipeline([
        ('model', KNeighborsClassifier())
    ]),
    'SVM': Pipeline([
        ('model', SVC(class_weight='balanced', probability=True, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    # Cross-validation on training data
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=None)

    # Fit on training data and predict on test data
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan

    results.append({
        'Model': name,
        'CV_F1_Mean': cv_scores.mean(),
        'CV_F1_Std': cv_scores.std(),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1,
        'ROC_AUC': roc_auc
    })

model_comparison = pd.DataFrame(results).sort_values(by='F1_Score', ascending=False)
print(model_comparison)
print("\nBest Model based on F1 Score:")
best_model_name = model_comparison.iloc[0]['Model'] 
print(best_model_name)  



Running preprocessing notebook: C:\Users\pradi\Pradip Workspace\The IOT acadamy assignment\AI ML Internship Programe\Telco Customer Churn\ipynb file\03_preprocessing.ipynb
Columns after dropping:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Engineered columns added: ['tenure_group', 'monthly_charge_per_tenure']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'monthly_charge_per_tenure']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'tenure_group']
Target labels: ['No' 'Yes']
Encoded

c:\Users\pradi\Pradip Workspace\The IOT acadamy assignment\AI ML Internship Programe\Telco Customer Churn\myenv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\pradi\Pradip Workspace\The IOT acadamy assignment\AI ML Internship Programe\Telco Customer Churn\myenv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\pradi\Pradip Workspace\The IOT acadamy assignment\AI ML Internship Programe\Telco Customer Churn\myenv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedCla

                 Model  CV_F1_Mean  CV_F1_Std  Accuracy  Precision    Recall  \
0  Logistic Regression    0.634421   0.020264  0.765791   0.541667  0.764706   
4        Random Forest    0.618500   0.023674  0.773598   0.562358  0.663102   
2                  SVM    0.601880   0.019040  0.728886   0.493103  0.764706   
3        Decision Tree    0.590606   0.013814  0.750887   0.526559  0.609626   
1                  KNN    0.540737   0.020041  0.765791   0.560440  0.545455   

   F1_Score   ROC_AUC  
0  0.634146  0.846571  
4  0.608589  0.829962  
2  0.599581  0.823462  
3  0.565056  0.705809  
1  0.552846  0.780738  

Best Model based on F1 Score:
Logistic Regression


2.5.3


In [ ]:
# Optional install (run once if needed)
# !pip install shap matplotlib seaborn


# ------------------------------------------------------------------
# 1) ROC curves for all 5 models on one chart
# ------------------------------------------------------------------
plt.figure(figsize=(10, 7))

for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    if y_proba is None:
        continue

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {roc_auc:.3f})",
        linewidth=2,
    )

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison for All Models")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 2) Confusion matrix for top 2 models
# ------------------------------------------------------------------
top2 = model_comparison.head(2)["Model"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model_name in zip(axes, top2):
    model = models[model_name]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Churn", "Churn"],
        yticklabels=["No Churn", "Churn"],
        cbar=False,
        ax=ax,
    )
    ax.set_title(f"{model_name} Confusion Matrix")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 3) Tune best model with RandomizedSearchCV
# ------------------------------------------------------------------
best_model_name = model_comparison.iloc[0]["Model"]
print(f"\nBest model by F1 Score: {best_model_name}")

best_model = models[best_model_name]

param_grid = {
    "Logistic Regression": {
        "model__C": [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20],
        "model__solver": ["liblinear", "lbfgs"],
        "model__penalty": ["l2"],
    },
    "KNN": {
        "model__n_neighbors": list(range(1, 31)),
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2],
    },
    "SVM": {
        "model__C": [0.1, 0.5, 1, 2, 5, 10, 20],
        "model__gamma": ["scale", "auto", 0.01, 0.1, 1],
        "model__kernel": ["rbf", "linear", "poly"],
    },
    "Decision Tree": {
        "model__max_depth": [None, 2, 3, 4, 5, 6, 8, 10, 12, 15],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__criterion": ["gini", "entropy"],
    },
    "Random Forest": {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [None, 5, 8, 10, 15, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", None],
        "model__bootstrap": [True, False],
    },
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=best_model,
    param_distributions=param_grid[best_model_name],
    n_iter=25,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)
best_tuned_model = random_search.best_estimator_

y_proba_tuned = best_tuned_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_proba_tuned)

print(f"\nTuned model: {best_model_name}")
print(f"Best params: {random_search.best_params_}")
print(f"Test ROC AUC: {test_auc:.4f}")

if test_auc < 0.80:
    print("\nWarning: Test ROC AUC is below target (0.80).")
    print("Try a broader search space or switch to a stronger model.")
else:
    print("\nTarget reached: Test ROC AUC >= 0.80")
    print("Aim is 0.85+, continue tuning if needed.")

# ------------------------------------------------------------------
# 4) SHAP summary plot + waterfall for one customer
# ------------------------------------------------------------------
# Use the model estimator inside the pipeline for SHAP
model_for_shap = best_tuned_model.named_steps["model"]

explainer = shap.Explainer(model_for_shap, X_train)
shap_values = explainer(X_test)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=X_train.columns.tolist(),
    max_display=15,
    show=False,
)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.show()

# Waterfall for one customer
customer_idx = 0
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[customer_idx], max_display=15)
plt.title(f"SHAP Waterfall for Customer #{customer_idx}")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 5) Save final model and model card
# ------------------------------------------------------------------
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

final_payload = {
    "model": best_tuned_model,
    "feature_names": list(X_train.columns),
    "best_model_name": best_model_name,
    "best_params": random_search.best_params_,
    "test_roc_auc": float(test_auc),
    "training_cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
}

with open(model_dir / "churn_model_v1.pkl", "wb") as f:
    pickle.dump(final_payload, f)

print(f"\nModel saved to: {model_dir / 'churn_model_v1.pkl'}")

model_card = f"""
# Customer Churn Prediction Model Card

## 1) Model Name
Churn Prediction Model v1

## 2) Objective
Predict whether a customer is likely to churn based on telecom usage and account attributes.

## 3) Dataset
Telco customer churn dataset, preprocessed and split into train/test sets.

## 4) Model Selected
- Best model (by F1 score): {best_model_name}
- Tuned via RandomizedSearchCV
- Final test ROC AUC: {test_auc:.4f}

## 5) Training Setup
- Train/Test split: as prepared in preprocessing notebook
- CV strategy: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- Objective metric: ROC AUC

## 6) Hyperparameters
{random_search.best_params_}

## 7) Evaluation Metrics
- Accuracy: {accuracy_score(y_test, best_tuned_model.predict(X_test)):.4f}
- Precision: {precision_score(y_test, best_tuned_model.predict(X_test), zero_division=0):.4f}
- Recall: {recall_score(y_test, best_tuned_model.predict(X_test), zero_division=0):.4f}
- F1 Score: {f1_score(y_test, best_tuned_model.predict(X_test), zero_division=0):.4f}
- ROC AUC: {test_auc:.4f}

## 8) Interpretability
- SHAP summary plot created for the final model
- SHAP waterfall plot generated for one customer example

## 9) Business Use
This model can support churn prevention campaigns by identifying at-risk customers earlier and prioritizing retention efforts.

## 10) Risks and Limitations
- Model performance depends on data quality and feature drift.
- Business context and fairness should be reviewed before deployment.
- If customer behavior changes, retraining is recommended.

## 11) Model Artifact
Saved as:
- `models/churn_model_v1.pkl`
"""

with open(model_dir / "model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print(f"Model card saved to: {model_dir / 'model_card.md'}")